# Clustering Analysis for Food Products NLP Dataset
## Comprehensive Guide to Product Clustering with SBERT, UMAP, and KMeans

This notebook performs a complete clustering analysis on food products using semantic embeddings and unsupervised learning.

## Setup & Configuration

In [1]:
# !pip install umap-learn

In [2]:
print("Hola")

Hola


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# NLP & ML libraries
from sentence_transformers import SentenceTransformer
import umap
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## Configuration Parameters

In [4]:
# File paths
DATA_DIR = Path("/home/alexl/TFM/data")
INPUT_FILE = DATA_DIR / "products_nlp_sample_5pct.csv"
PLOTS_DIR = DATA_DIR / "clustering_plots"
RESULTS_DIR = DATA_DIR / "clustering_results"

# Create directories
PLOTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Clustering parameters
EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # Fast & lightweight
N_CLUSTERS = 12  # Change this to experiment
BATCH_SIZE = 32
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

print(f"📁 Data dir: {DATA_DIR}")
print(f"📊 Input: {INPUT_FILE}")
print(f"📈 Output dirs: {PLOTS_DIR}, {RESULTS_DIR}")
print(f"\n🔧 Configuration:")
print(f"   Embedding model: {EMBEDDING_MODEL}")
print(f"   Number of clusters: {N_CLUSTERS}")

📁 Data dir: /home/alexl/TFM/data
📊 Input: /home/alexl/TFM/data/products_nlp_sample_5pct.csv
📈 Output dirs: /home/alexl/TFM/data/clustering_plots, /home/alexl/TFM/data/clustering_results

🔧 Configuration:
   Embedding model: all-MiniLM-L6-v2
   Number of clusters: 12


## 1. Load and Explore Data

In [5]:
# Load data
print(f"Loading data from {INPUT_FILE}...")
df = pd.read_csv("data/products_nlp_full_sample_5pct.csv")

print(f"✓ Loaded {len(df):,} rows")
print(f"\nDataframe shape: {df.shape}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

Loading data from /home/alexl/TFM/data/products_nlp_sample_5pct.csv...
✓ Loaded 57,238 rows

Dataframe shape: (57238, 13)

Columns: code, lang_product_name, lang_ingredients_text, product_name_clean, generic_name_clean, categories_clean, food_groups_clean, ingredients_text_clean, nlp_text, nlp_word_count, approx_tokens, nutrition-score-fr_100g, created_datetime

Data types:
code                        object
lang_product_name           object
lang_ingredients_text       object
product_name_clean          object
generic_name_clean          object
categories_clean            object
food_groups_clean           object
ingredients_text_clean      object
nlp_text                    object
nlp_word_count               int64
approx_tokens                int64
nutrition-score-fr_100g    float64
created_datetime            object
dtype: object

Missing values:
code                           0
lang_product_name           9730
lang_ingredients_text       5830
product_name_clean           290
gener

In [6]:
# Display sample data
print("Sample products:")
df[['code', 'lang_product_name', 'nlp_text']].head(3)

Sample products:


,code,lang_product_name,nlp_text
0,0007078405636,en,"Pancakes, original. Whey, enriched wheat flour..."
1,0007078405666,en,"Pizza bites, cheese. meals, pizzas pies and qu..."
2,0007078405668,en,"Combination pizza bites. meals, pizzas pies an..."


In [7]:
# Basic statistics
print("NLP Text Statistics:")
word_counts = df['nlp_text'].str.split().str.len()
print(f"  Mean length: {word_counts.mean():.1f} words")
print(f"  Median length: {word_counts.median():.1f} words")
print(f"  Min length: {word_counts.min()} words")
print(f"  Max length: {word_counts.max()} words")

print(f"\nLanguage distribution:")
print(df['lang_product_name'].value_counts().head(10))

NLP Text Statistics:
  Mean length: 46.5 words
  Median length: 42.0 words
  Min length: 1.0 words
  Max length: 204.0 words

Language distribution:
lang_product_name
en    24398
fr     9221
de     3105
it     2926
es     1427
no      815
nl      717
da      512
sv      442
fi      433
Name: count, dtype: int64


## 2. Generate Sentence Embeddings

Using SBERT (Sentence-BERT) to convert product text into semantic embeddings. This captures the meaning of product descriptions in a dense vector space.

In [8]:
# Load embedding model
print(f"Loading {EMBEDDING_MODEL} model...")
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"✓ Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading all-MiniLM-L6-v2 model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded. Embedding dimension: 384


In [9]:
# Generate embeddings
print(f"\nGenerating embeddings for {len(df):,} products...")
embeddings = model.encode(
    df['nlp_text'].tolist(),
    show_progress_bar=True,
    batch_size=BATCH_SIZE
)

print(f"\n✓ Embeddings generated!")
print(f"  Shape: {embeddings.shape}")
print(f"  Data type: {embeddings.dtype}")
print(f"  Memory: {embeddings.nbytes / 1e6:.1f} MB")


Generating embeddings for 57,238 products...


Batches:   0%|          | 0/1789 [00:00<?, ?it/s]


✓ Embeddings generated!
  Shape: (57238, 384)
  Data type: float32
  Memory: 87.9 MB


## 3. Dimensionality Reduction with UMAP

Reduce 384-dimensional embeddings to 2D for visualization while preserving meaningful relationships.

In [10]:
# Apply UMAP
print(f"Reducing to 2D using UMAP...")
reducer = umap.UMAP(
    n_components=2,
    random_state=42,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    verbose=1
)
embeddings_2d = reducer.fit_transform(embeddings)

print(f"\n✓ Dimensionality reduction complete!")
print(f"  Original shape: {embeddings.shape}")
print(f"  Reduced shape: {embeddings_2d.shape}")

Reducing to 2D using UMAP...
UMAP(n_jobs=1, random_state=42, verbose=1)
Fri Mar 27 12:04:33 2026 Construct fuzzy simplicial set
Fri Mar 27 12:04:33 2026 Finding Nearest Neighbors
Fri Mar 27 12:04:33 2026 Building RP forest with 17 trees
Fri Mar 27 12:04:40 2026 NN descent for 16 iterations
	 1  /  16
	 2  /  16
	 3  /  16
	 4  /  16
	 5  /  16
	Stopping threshold met -- exiting after 5 iterations
Fri Mar 27 12:04:53 2026 Finished Nearest Neighbor Search
Fri Mar 27 12:04:57 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Fri Mar 27 12:05:28 2026 Finished embedding

✓ Dimensionality reduction complete!
  Original shape: (57238, 384)
  Reduced shape: (57238, 2)


## 4. Find Optimal Number of Clusters (Optional)

Use the elbow method to automatically find the best k value.

In [ ]:
# Elbow method to find optimal k
print("Finding optimal number of clusters (this may take a while)...\n")

max_clusters = 20
inertias = []
silhouette_scores = []
k_range = range(2, max_clusters + 1)

for k in k_range:
    print(f"Testing k={k:2d}...", end=" ", flush=True)
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_test = kmeans_test.fit_predict(embeddings)
    inertia = kmeans_test.inertia_
    silhouette = silhouette_score(embeddings, labels_test)
    inertias.append(inertia)
    silhouette_scores.append(silhouette)
    print(f"inertia={inertia:.2f}, silhouette={silhouette:.4f}")

print("\n✓ Optimal cluster analysis complete")

Finding optimal number of clusters (this may take a while)...

Testing k= 2... 

In [ ]:
# Plot elbow curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=N_CLUSTERS, color='red', linestyle='--', label=f'Selected k={N_CLUSTERS}')
ax1.set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Inertia', fontsize=12, fontweight='bold')
ax1.set_title('Elbow Method: Finding Optimal k', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax2.axvline(x=N_CLUSTERS, color='red', linestyle='--', label=f'Selected k={N_CLUSTERS}')
ax2.set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
ax2.set_title('Silhouette Score by k', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / "elbow_method.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Elbow plot saved")

## 5. Perform KMeans Clustering

In [ ]:
# Perform clustering
print(f"Performing KMeans clustering with k={N_CLUSTERS}...")
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10, verbose=1)
labels = kmeans.fit_predict(embeddings)

silhouette_avg = silhouette_score(embeddings, labels)
print(f"\n✓ Clustering complete!")
print(f"  Silhouette Score: {silhouette_avg:.4f}")

In [ ]:
# Cluster distribution
print(f"\nCluster Distribution:")
print("="*50)
unique, counts = np.unique(labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    pct = count / len(labels) * 100
    bar = "█" * int(pct / 2)
    print(f"Cluster {cluster_id:2d}: {count:6,} products ({pct:5.1f}%) {bar}")

## 6. Visualize Clusters

In [ ]:
# Main cluster scatter plot
fig, ax = plt.subplots(figsize=(16, 12))

scatter = ax.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1],
    c=labels, 
    cmap='tab20' if N_CLUSTERS <= 20 else 'hsv',
    alpha=0.6, 
    s=40, 
    edgecolors='black', 
    linewidth=0.5
)

cbar = plt.colorbar(scatter, ax=ax, label='Cluster ID')
cbar.ax.tick_params(labelsize=10)

ax.set_xlabel('UMAP Dimension 1', fontsize=12, fontweight='bold')
ax.set_ylabel('UMAP Dimension 2', fontsize=12, fontweight='bold')
ax.set_title(f'Product Clustering Overview ({N_CLUSTERS} Clusters | Silhouette: {silhouette_avg:.4f})',
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(PLOTS_DIR / f"clusters_{N_CLUSTERS}.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Cluster visualization saved")

In [ ]:
# Silhouette plot
silhouette_vals = silhouette_samples(embeddings, labels)

fig, ax = plt.subplots(figsize=(10, N_CLUSTERS * 0.5))

y_lower = 10
colors = plt.cm.tab20(np.linspace(0, 1, N_CLUSTERS))

for i in range(N_CLUSTERS):
    cluster_silhouette_vals = silhouette_vals[labels == i]
    cluster_silhouette_vals.sort()
    
    size_cluster_i = cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0, cluster_silhouette_vals,
        facecolor=colors[i], edgecolor=colors[i], alpha=0.7
    )
    
    y_lower = y_upper + 10

ax.axvline(x=silhouette_avg, color="red", linestyle="--", linewidth=2,
           label=f'Average: {silhouette_avg:.3f}')
ax.set_xlabel('Silhouette Score', fontsize=12, fontweight='bold')
ax.set_ylabel('Cluster Label', fontsize=12, fontweight='bold')
ax.set_title(f'Silhouette Plot for {N_CLUSTERS} Clusters', fontsize=13, fontweight='bold')
ax.set_yticks([])
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.2, axis='x')

plt.tight_layout()
plt.savefig(PLOTS_DIR / f"silhouette_{N_CLUSTERS}.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Silhouette plot saved")

## 7. Cluster Profiling

In [ ]:
# Create profiling dataframe
profile_data = []

for cluster_id in range(N_CLUSTERS):
    cluster_df = df[labels == cluster_id]
    
    profile = {
        'cluster_id': cluster_id,
        'n_products': len(cluster_df),
        'pct_of_total': len(cluster_df) / len(df) * 100,
    }
    
    # Language
    if 'lang_product_name' in cluster_df.columns:
        lang_dist = cluster_df['lang_product_name'].value_counts()
        profile['dominant_language'] = lang_dist.index[0] if len(lang_dist) > 0 else 'N/A'
        profile['n_languages'] = len(lang_dist)
    
    # Nutrition
    if 'nutrition-score-fr_100g' in cluster_df.columns:
        nutrition = cluster_df['nutrition-score-fr_100g'].dropna()
        if len(nutrition) > 0:
            profile['avg_nutrition_score'] = nutrition.mean()
            profile['nutrition_score_std'] = nutrition.std()
    
    # Text stats
    if 'nlp_text' in cluster_df.columns:
        word_counts = cluster_df['nlp_text'].str.split().str.len()
        profile['avg_text_length'] = word_counts.mean()
        profile['median_text_length'] = word_counts.median()
    
    profile_data.append(profile)

profile_df = pd.DataFrame(profile_data)
profile_df.to_csv(RESULTS_DIR / f"cluster_profile_{N_CLUSTERS}.csv", index=False)

print("Cluster Profiles:")
print(profile_df.to_string(index=False))

## 8. Sample Products Per Cluster

In [ ]:
# Display sample products from each cluster
df_with_cluster = df.copy()
df_with_cluster['cluster'] = labels

for cluster_id in range(N_CLUSTERS):
    cluster_df = df_with_cluster[df_with_cluster['cluster'] == cluster_id]
    print(f"\n{'─'*80}")
    print(f"Cluster {cluster_id} ({len(cluster_df):,} products)")
    print(f"{'─'*80}")
    
    samples = cluster_df[['code', 'lang_product_name', 'nlp_text']].head(3)
    
    for idx, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"\n  {idx}. [{row['code']}] {row['lang_product_name']}")
        nlp_preview = row['nlp_text'][:100] + "..." if len(str(row['nlp_text'])) > 100 else row['nlp_text']
        print(f"     {nlp_preview}")

## 9. Export Results

In [ ]:
# Create full results dataframe
results_df = df.copy()
results_df['cluster'] = labels
results_df['umap_x'] = embeddings_2d[:, 0]
results_df['umap_y'] = embeddings_2d[:, 1]

# Save full results
results_path = RESULTS_DIR / f"clustering_results_{N_CLUSTERS}_full.csv"
results_df.to_csv(results_path, index=False)
print(f"✓ Full results saved: {results_path}")
print(f"  Shape: {results_df.shape}")

# Save embeddings
embeddings_path = RESULTS_DIR / f"embeddings_{N_CLUSTERS}.npy"
np.save(embeddings_path, embeddings)
print(f"\n✓ Embeddings saved: {embeddings_path}")
print(f"  Shape: {embeddings.shape}")

In [ ]:
# Summary
print("\n" + "="*80)
print("CLUSTERING ANALYSIS COMPLETE!")
print("="*80)
print(f"\n📊 Summary:")
print(f"  Total products analyzed: {len(df):,}")
print(f"  Number of clusters: {N_CLUSTERS}")
print(f"  Silhouette Score: {silhouette_avg:.4f}")
print(f"  Embedding dimension: {embeddings.shape[1]}")
'''
print(f"\n📁 Output Files:")
print(f"  📈 Plots: {PLOTS_DIR}")
for plot in sorted(PLOTS_DIR.glob("*.png")):
    print(f"     - {plot.name}")

print(f"\n  📊 Results: {RESULTS_DIR}")
for result in sorted(RESULTS_DIR.glob("*_*.csv")):
    print(f"     - {result.name}")
for npy in sorted(RESULTS_DIR.glob("*.npy")):
    print(f"     - {npy.name}")

print(f"\n✅ Next Steps:")
print(f"  1. Review visualizations in {PLOTS_DIR}")
print(f"  2. Analyze cluster profiles from the CSV results")
print(f"  3. Use embeddings for downstream tasks (search, classification, etc.)")
print(f"  4. Experiment with different k values by changing N_CLUSTERS")
'''